Libraries imports

In [0]:
import pyspark.sql.functions as F

Read, transformations and join

In [0]:

df_sales = spark.table("abinbev_case_bronze.beverage_sales")
df_channel = spark.table("abinbev_case_bronze.beverage_channel_group")

df_sales_clean = (
    df_sales
    .select(
        F.to_date(F.trim(F.col("date")), "M/d/yyyy").alias("sale_date"),
        F.trim(F.upper(F.col("brand_nm"))).alias("brand_nm"),
        F.col("ce_brand_flvr").alias("brand_flvr_id"),
        F.trim(F.upper(F.col("btlr_org_lvl_c_desc"))).alias("btlr_org_lvl_c_desc"),
        F.trim(F.upper(F.col("chnl_group"))).alias("chnl_group"),
        F.trim(F.upper(F.col("trade_chnl_desc"))).alias("trade_chnl_desc"),
        F.trim(F.upper(F.col("pkg_cat"))).alias("pkg_cat"),
        F.trim(F.upper(F.col("pkg_cat_desc"))).alias("pkg_cat_desc"),
        F.trim(F.upper(F.col("tsr_pckg_nm"))).alias("tsr_pckg_nm"),
        F.col("_volume").alias("volume"),
        F.col("period").cast("int").alias("period")
    )
)

df_sales_enriched = (
    df_sales_clean.alias("s")
    .join(
        df_channel.alias("c"),
        F.trim(F.upper(F.col("s.trade_chnl_desc"))) == F.trim(F.upper(F.col("c.TRADE_CHNL_DESC"))),
        "left"
    )
    .select(
        "s.*",
        F.trim(F.upper(F.col("c.TRADE_GROUP_DESC"))).alias("trade_group_desc"),
        F.trim(F.upper(F.col("c.TRADE_TYPE_DESC"))).alias("trade_type_desc")
    )
)

df_sales_final = (
    df_sales_enriched
    .withColumn("year_partition", F.year(F.col("sale_date")))
    .withColumn("month_partition", F.month(F.col("sale_date")))
    .withColumn("day_partition", F.dayofmonth(F.col("sale_date")))
)

Save as silver delta table

In [0]:
df_sales_final.write.mode("overwrite").format("delta").partitionBy("year_partition", "month_partition", "day_partition").saveAsTable(
    "abinbev_case_silver.fact_sales")

display(df_sales_final)

In [0]:
df_sales_final.count()